In [ ]:
import sys, math, traceback
from pathlib import Path
sys.path.insert(0, str(Path('/home/binghin2/Myproject/Research/CATSA/Train/Individual_data/Temp')))

import numpy as np
import pandas as pd
from copy import deepcopy
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, roc_auc_score, confusion_matrix)
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

torch.set_float32_matmul_precision('high')
torch.backends.cudnn.benchmark = True
USE_BF16 = torch.cuda.is_available()
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  |  PyTorch: {torch.__version__}  |  bf16: {USE_BF16}')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
TASKS        = ['Baseline', 'Logic', 'Nback', 'Stroop', 'Sudoku']
STRESS_TASKS = {'Logic', 'Nback', 'Stroop', 'Sudoku'}
FS           = 4        # Temp / EDA sampling rate
SIGNAL       = 'TEMP'   # change to 'EDA' for EDA channel

CATSA_ROOT  = Path('/home/binghin2/Myproject/Dataset/CATSA')
E4_ROOT     = Path('/home/binghin2/Myproject/Dataset/EmpaticaE4Stress/Subjects')
SUBJECTS_E4 = [f'subject_{i:02d}' for i in range(1, 7)]
SAVE_DIR    = Path('/home/binghin2/Myproject/Research/CATSA/Train/Individual_data/Temp/Save_model_TenModels')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# From-scratch models (bf16 OK) + foundation model (Chronos2, float32 only)
MODELS = ['TimeMixerPP', 'Medformer', 'TSLANet', 'ModernTCN', 'CrossGNN',
          'TimesNet', 'Mamba2', 'iTransformer', 'PatchTST', 'Chronos2']

# Base HP shared by all from-scratch models
HP = {
    'seed': 42, 'val_ratio': 0.10,
    'window_size': FS * 60,   # 240
    'stride':      FS * 10,   # 40
    'batch_size':  64,
    'epochs':      50,
    'lr':          1e-4,
    'wd':          1e-4,
    'patience':    8,
    'dropout':     0.1,
}

# Chronos-2 overrides (foundation model: smaller lr, smaller batch, fewer epochs)
CHRONOS_HP = {
    'batch_size': 32, 'epochs': 30, 'lr': 5e-5,
    'wd': 1e-4, 'patience': 6, 'dropout': 0.3,
}

print(f"W={HP['window_size']}, S={HP['stride']}, n_models={len(MODELS)}")
print(f'Models: {MODELS}')
print(f'Save: {SAVE_DIR}')

In [ ]:
# ── Data utilities ────────────────────────────────────────────────────────────
def discover_subjects(root):
    out = []
    for d in sorted(root.glob('Sub*'), key=lambda p: int(p.name[3:])):
        if d.is_dir() and all((d / t / f'{SIGNAL}.csv').exists() for t in TASKS):
            out.append(d.name)
    return out

def read_signal(path):
    df = pd.read_csv(path)
    nc = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    v  = df[nc[0]].to_numpy(np.float32) if nc else pd.to_numeric(df.iloc[:, 0], errors='coerce').to_numpy(np.float32)
    return v[~np.isnan(v)]

def make_windows(sig, W, S):
    if len(sig) < W: return np.empty((0, 1, W), np.float32)
    return np.asarray([sig[i:i+W] for i in range(0, len(sig)-W+1, S)], np.float32)[:, None, :]

def build_catsa_arrays(subjects, W, S):
    xs, ys = [], []
    for sub in subjects:
        path = CATSA_ROOT / sub
        all_v = np.concatenate([read_signal(path / t / f'{SIGNAL}.csv') for t in TASKS])
        mu = float(np.mean(all_v)); sigma = float(np.std(all_v)) + 1e-8
        for task in TASKS:
            norm = (read_signal(path / task / f'{SIGNAL}.csv') - mu) / sigma
            w = make_windows(norm, W, S)
            if len(w): xs.append(w); ys.append(np.full(len(w), int(task in STRESS_TASKS), np.int64))
    return np.concatenate(xs), np.concatenate(ys)

def build_e4_labels(n, fs=4):
    fixed_s = {'rest0':180,'task1':600,'rest1':120,'task2':300,'rest2':120,
               'task3':180,'rest3':120,'rest4':120,'task5':60,'rest5':120}
    task4_s = max(0, n // fs - sum(fixed_s.values()))
    segs = [('rest0',180,0),('task1',600,1),('rest1',120,-1),('task2',300,1),
            ('rest2',120,-1),('task3',180,1),('rest3',120,-1),('task4',task4_s,1),
            ('rest4',120,-1),('task5',60,1),('rest5',120,-1)]
    labels = np.full(n, -1, np.int64); cur = 0
    for _, dur_s, lab in segs:
        end = min(cur + dur_s * fs, n)
        if end > cur: labels[cur:end] = lab
        cur = end
        if cur >= n: break
    return labels, int(task4_s)

def e4_windows(sig, labels, W, S):
    mask = labels >= 0
    mu    = float(np.mean(sig[mask])) if mask.any() else float(np.mean(sig))
    sigma = (float(np.std(sig[mask])) + 1e-8) if mask.any() else (float(np.std(sig)) + 1e-8)
    norm  = ((sig - mu) / sigma).astype(np.float32)
    xs, ys = [], []
    for i in range(0, len(norm) - W + 1, S):
        u = np.unique(labels[i:i+W])
        if len(u) == 1 and u[0] in (0, 1): xs.append(norm[i:i+W]); ys.append(int(u[0]))
    if not xs: return np.empty((0, 1, W), np.float32), np.empty(0, np.int64)
    return np.asarray(xs, np.float32)[:, None, :], np.asarray(ys, np.int64)

print('Data utilities loaded.')

In [ ]:
# ── Training utilities ────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None): super().__init__(); self.gamma, self.alpha = gamma, alpha
    def forward(self, logits, y):
        y = y.float(); bce = F.binary_cross_entropy_with_logits(logits, y, reduction='none')
        pt = torch.sigmoid(logits) * y + (1 - torch.sigmoid(logits)) * (1 - y)
        fl = (1 - pt).pow(self.gamma) * bce
        if self.alpha is not None: fl = (self.alpha * y + (1 - self.alpha) * (1 - y)) * fl
        return fl.mean()

def make_loader(x, y, bs, shuffle):
    return DataLoader(TensorDataset(torch.from_numpy(x), torch.from_numpy(y).float()),
                      batch_size=bs, shuffle=shuffle, num_workers=4, pin_memory=True)

@torch.no_grad()
def evaluate(model, loader, criterion, use_bf16=True):
    model.eval(); tot = n = tp = tn = fp = fn = 0; all_prob = []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=use_bf16):
            lg = model(xb).float()
        tot += criterion(lg, yb).item() * len(xb); n += len(xb)
        prob = torch.sigmoid(lg).cpu().numpy(); all_prob.extend(prob.tolist())
        pred = (prob >= 0.5).astype(int); yi = yb.long().cpu().numpy()
        tp += int(((pred==1)&(yi==1)).sum()); tn += int(((pred==0)&(yi==0)).sum())
        fp += int(((pred==1)&(yi==0)).sum()); fn += int(((pred==0)&(yi==1)).sum())
    acc = (tp+tn)/max(n,1); pre = tp/max(tp+fp,1); rec = tp/max(tp+fn,1)
    return {'loss': tot/max(n,1), 'accuracy': acc,
            'f1': 2*pre*rec/max(pre+rec,1e-8), 'precision': pre, 'recall': rec,
            'probs': all_prob}

def train_epoch(model, loader, optimizer, criterion, use_bf16=True):
    model.train(); tot = n = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=use_bf16):
            loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        tot += loss.item() * len(xb); n += len(xb)
    return tot / max(n, 1)

@torch.no_grad()
def infer_chunks(model, x_np, chunk_size=256, use_bf16=True):
    """Batched inference to avoid OOM for large models (e.g. Chronos-2)."""
    probs = []
    for i in range(0, len(x_np), chunk_size):
        xb = torch.from_numpy(x_np[i:i+chunk_size]).to(DEVICE)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=use_bf16):
            lg = model(xb).float()
        probs.append(torch.sigmoid(lg).cpu().numpy())
    return np.concatenate(probs) if probs else np.array([])

print('Training utilities loaded.')

In [ ]:
# ── Import models & sanity-check shapes ──────────────────────────────────────
from models_six import (
    TimeMixerPPClassifier, MedformerClassifier, TSLANetClassifier,
    ModernTCNClassifier, CrossGNNClassifier, TimesNetClassifier,
    Mamba2Classifier, iTransformerClassifier, PatchTSTClassifier,
    Chronos2Classifier, build_model
)

W = HP['window_size']
SCRATCH_MODELS = [m for m in MODELS if m != 'Chronos2']
print(f'{"Model":<14}  {"Output":>12}  {"Params":>12}')
print('-' * 44)
for name in SCRATCH_MODELS:
    m = build_model(name, W=W, n_channels=1)
    x = torch.randn(2, 1, W)
    with torch.no_grad(): out = m(x)
    p = sum(pp.numel() for pp in m.parameters() if pp.requires_grad)
    print(f'{name:<14}  {str(out.shape):>12}  {p:>12,}')
    del m, x, out
print('(Chronos2 checked separately after pipeline load)')

In [ ]:
# ── Chronos-2 pipeline loading (skip cell if not installed) ───────────────────
# pip install chronos-forecasting
chronos_pipeline = None

if 'Chronos2' in MODELS:
    try:
        from chronos import ChronosPipeline
        import warnings; warnings.filterwarnings('ignore', category=FutureWarning)
        print('Loading amazon/chronos-t5-small ...')
        chronos_pipeline = ChronosPipeline.from_pretrained(
            'amazon/chronos-t5-small', device_map='cpu', dtype=torch.float32)
        # Quick shape check
        m_c = build_model('Chronos2', W=W, n_channels=1, pipeline=chronos_pipeline).to(DEVICE)
        with torch.no_grad():
            out_c = m_c(torch.randn(2, 1, W).to(DEVICE))
        p_c = sum(pp.numel() for pp in m_c.parameters() if pp.requires_grad)
        print(f'Chronos2       {str(out_c.shape):>12}  {p_c:>12,}')
        del m_c, out_c
    except ImportError:
        print('WARNING: chronos-forecasting not installed. Removing Chronos2 from MODELS.')
        MODELS = [m for m in MODELS if m != 'Chronos2']
    except Exception as e:
        print(f'WARNING: Chronos-2 failed to load ({e}). Removing from MODELS.')
        MODELS = [m for m in MODELS if m != 'Chronos2']

print(f'\nFinal model list ({len(MODELS)}): {MODELS}')

In [ ]:
# ── Load CATSA data once (shared across all models) ───────────────────────────
np.random.seed(HP['seed']); torch.manual_seed(HP['seed'])
if torch.cuda.is_available(): torch.cuda.manual_seed_all(HP['seed'])

subjects = discover_subjects(CATSA_ROOT)
print(f'Total CATSA subjects: {len(subjects)}')

rng = np.random.default_rng(HP['seed']); idx = rng.permutation(len(subjects))
n_val = max(4, int(len(subjects) * HP['val_ratio']))
val_subs   = [subjects[i] for i in sorted(idx[:n_val])]
train_subs = [subjects[i] for i in sorted(idx[n_val:])]
print(f'Train: {len(train_subs)} subjects | Val: {len(val_subs)} subjects')

W, S = HP['window_size'], HP['stride']
print('Building CATSA windows...')
x_train, y_train = build_catsa_arrays(train_subs, W, S)
x_val,   y_val   = build_catsa_arrays(val_subs,   W, S)
print(f'Train: {x_train.shape}  Val: {x_val.shape}')

pos   = float(y_train.sum())
alpha = float((len(y_train) - pos) / len(y_train))
print(f'Stress: {pos:.0f}/{len(y_train)} ({pos/len(y_train)*100:.1f}%)  alpha={alpha:.3f}')

In [ ]:
# ── Main loop: train each model, eval on E4, save checkpoint & CSV ────────────
results_all = {}

def run_one_model(model_name):
    is_chronos = model_name == 'Chronos2'
    use_bf16   = USE_BF16 and not is_chronos   # Chronos uses float32 internally
    hp = {**HP, **(CHRONOS_HP if is_chronos else {})}

    print(f"\n{'='*60}\n  {model_name}{'  [fine-tune]' if is_chronos else '  [from-scratch]'}\n{'='*60}")

    loader_tr = make_loader(x_train, y_train, hp['batch_size'], True)
    loader_va = make_loader(x_val,   y_val,   hp['batch_size'], False)
    criterion = FocalLoss(gamma=2.0, alpha=alpha).to(DEVICE)

    # Build model
    if is_chronos:
        model = build_model(model_name, W=W, n_channels=1, pipeline=chronos_pipeline).to(DEVICE)
    else:
        model = build_model(model_name, W=W, n_channels=1).to(DEVICE)
    p_total = sum(pp.numel() for pp in model.parameters() if pp.requires_grad)
    print(f'  Trainable params: {p_total:,}')

    # Optimizer (Chronos: split lr for encoder vs head)
    if is_chronos:
        enc_p  = [p for p in model.chron_model.parameters() if p.requires_grad]
        head_p = list(model.head.parameters())
        optimizer = torch.optim.Adam(
            [{'params': enc_p, 'lr': hp['lr'] * 0.1},
             {'params': head_p, 'lr': hp['lr']}], weight_decay=hp['wd'])
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=hp['lr'], weight_decay=hp['wd'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=4)

    best_val_loss = float('inf'); best_state = None; best_epoch = 0; wait = 0; history = []

    for epoch in range(1, hp['epochs'] + 1):
        tr_loss = train_epoch(model, loader_tr, optimizer, criterion, use_bf16)
        val_m   = evaluate(model, loader_va, criterion, use_bf16)
        scheduler.step(val_m['loss'])
        history.append({'epoch': epoch, 'train_loss': tr_loss,
                        'val_loss': val_m['loss'], 'val_f1': val_m['f1']})
        if val_m['loss'] < best_val_loss:
            best_val_loss = val_m['loss']; best_epoch = epoch; wait = 0
            best_state = deepcopy(model.state_dict())
            torch.save({'model_state_dict': best_state, 'hp': hp,
                        'epoch': epoch, 'model': model_name},
                       SAVE_DIR / f'best_model_{model_name}.pt')
        else:
            wait += 1
            if wait >= hp['patience']:
                print(f'  Early stop @ epoch {epoch}  (best: {best_epoch})')
                break
        if epoch % 5 == 0 or epoch == 1:
            print(f'  [{epoch:3d}] train={tr_loss:.4f}  val={val_m["loss"]:.4f}  f1={val_m["f1"]:.4f}')

    model.load_state_dict(best_state)
    print(f'  Best epoch: {best_epoch}  val_loss: {best_val_loss:.4f}')

    # ── Evaluate on EmpaticaE4Stress ──────────────────────────────────────────
    model.eval()
    chunk_size = 16 if is_chronos else 256   # small chunks for large foundation model
    all_true, all_pred, all_prob = [], [], []; rows = []

    for sub in SUBJECTS_E4:
        sig = read_signal(E4_ROOT / sub / f'{SIGNAL}.csv')
        labels, task4_sec = build_e4_labels(len(sig), fs=FS)
        x, y_true = e4_windows(sig, labels, W, S)
        if len(x) == 0:
            rows.append({'subject': sub, 'n_windows': 0, 'accuracy': float('nan'),
                         'f1': float('nan'), 'precision': float('nan'), 'recall': float('nan')})
            continue
        prob   = infer_chunks(model, x, chunk_size=chunk_size, use_bf16=use_bf16)
        y_pred = (prob >= 0.5).astype(int)
        rows.append({'subject': sub, 'task4_sec': task4_sec, 'n_windows': len(y_true),
                     'accuracy':   float(accuracy_score(y_true, y_pred)),
                     'f1':         float(f1_score(y_true, y_pred, zero_division=0)),
                     'precision':  float(precision_score(y_true, y_pred, zero_division=0)),
                     'recall':     float(recall_score(y_true, y_pred, zero_division=0))})
        all_true.extend(y_true.tolist()); all_pred.extend(y_pred.tolist()); all_prob.extend(prob.tolist())

    display(pd.DataFrame(rows))

    if not all_true:
        return {'model': model_name, 'accuracy': float('nan'), 'f1': float('nan'),
                'auroc': float('nan'), 'precision': float('nan'), 'recall': float('nan'),
                'best_epoch': best_epoch, 'best_val_loss': best_val_loss,
                'history': history, 'subject_rows': rows}

    ov_acc  = accuracy_score(all_true, all_pred)
    ov_f1   = f1_score(all_true, all_pred, zero_division=0)
    ov_prec = precision_score(all_true, all_pred, zero_division=0)
    ov_rec  = recall_score(all_true, all_pred, zero_division=0)
    try:    auroc = roc_auc_score(all_true, all_prob)
    except: auroc = float('nan')
    print(f'  Overall — Acc:{ov_acc:.4f}  F1:{ov_f1:.4f}  AUROC:{auroc:.4f}  Prec:{ov_prec:.4f}  Rec:{ov_rec:.4f}')
    cm = confusion_matrix(all_true, all_pred); print(f'  CM: {cm.tolist()}')

    return {'model': model_name, 'accuracy': ov_acc, 'f1': ov_f1, 'auroc': auroc,
            'precision': ov_prec, 'recall': ov_rec,
            'best_epoch': best_epoch, 'best_val_loss': best_val_loss,
            'history': history, 'subject_rows': rows}


for model_name in MODELS:
    try:
        result = run_one_model(model_name)
        results_all[model_name] = result
    except Exception as e:
        print(f'\n[{model_name}] FAILED: {e}')
        traceback.print_exc()
        results_all[model_name] = {'model': model_name, 'error': str(e),
                                   'accuracy': float('nan'), 'f1': float('nan'),
                                   'auroc': float('nan'), 'precision': float('nan'), 'recall': float('nan')}

    # Intermediate CSV after each model
    prog = pd.DataFrame([{k: v for k, v in r.items() if k not in ('history', 'subject_rows')}
                         for r in results_all.values()])
    prog.to_csv(SAVE_DIR / 'results_progress.csv', index=False)
    print(f'  [progress saved → results_progress.csv]')

print('\n>>> All models done!')

In [ ]:
# ── Final results table ───────────────────────────────────────────────────────
cols = ['accuracy', 'f1', 'auroc', 'precision', 'recall', 'best_epoch', 'best_val_loss']
final_df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('history', 'subject_rows', 'error')}
    for r in results_all.values()
]).set_index('model')[cols].sort_values('f1', ascending=False)

print('=== Final Results — 10 Models (sorted by F1) ===')
display(final_df.style
        .format({c: '{:.4f}' for c in ['accuracy','f1','auroc','precision','recall','best_val_loss']},
                na_rep='N/A')
        .background_gradient(cmap='YlGn', subset=['accuracy', 'f1', 'auroc']))

final_df.to_csv(SAVE_DIR / 'final_results.csv')
print(f'Saved → {SAVE_DIR}/final_results.csv')

In [ ]:
# ── Visualization ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Grouped bar: Acc / F1 / AUROC
model_names = final_df.index.tolist(); n_m = len(model_names)
x_pos = np.arange(n_m); width = 0.25
for i, (metric, color) in enumerate(zip(['accuracy','f1','auroc'],
                                         ['#4C78A8','#F58518','#54A24B'])):
    axes[0].bar(x_pos + i*width, final_df[metric].values, width,
                label=metric.upper(), color=color, alpha=0.85)
axes[0].set_xticks(x_pos + width); axes[0].set_xticklabels(model_names, rotation=40, ha='right', fontsize=8)
axes[0].set_ylim(0, 1); axes[0].legend(fontsize=8); axes[0].set_title('Overall Metrics')
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# Val-loss learning curves
colors_lc = plt.cm.tab10(np.linspace(0, 1, len(results_all)))
for (name, r), color in zip(results_all.items(), colors_lc):
    if 'history' in r and r['history']:
        h = r['history']
        axes[1].plot([e['epoch'] for e in h], [e['val_loss'] for e in h],
                     label=name, color=color, linewidth=1.6)
axes[1].set_title('Val Loss Curves'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Loss')
axes[1].legend(fontsize=7); axes[1].grid(linestyle='--', alpha=0.4)

# Horizontal F1 bar (ranked best→worst, bottom→top)
f1_vals = final_df['f1'].values[::-1]; names_sorted = model_names[::-1]
bar_colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, n_m))
axes[2].barh(range(n_m), f1_vals, color=bar_colors)
axes[2].set_yticks(range(n_m)); axes[2].set_yticklabels(names_sorted, fontsize=8)
axes[2].set_xlim(0, 1); axes[2].axvline(0.5, color='r', linestyle='--', alpha=0.5)
axes[2].set_title('F1 Ranking'); axes[2].grid(axis='x', linestyle='--', alpha=0.4)
for i, v in enumerate(f1_vals):
    if not np.isnan(v): axes[2].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=8)

fig.suptitle(f'10-Model Comparison — CATSA→EmpaticaE4 ({SIGNAL}, {FS}Hz)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {SAVE_DIR}/final_comparison.png')